In [0]:
# sel_projects = all searches the entire project portfolio whereas sel_projects = selected will search only the manually uploaded project list
sel_projects = 'all'

# sel_hierarchy can be 'decentralization' or 'pfm' or 'asset management'
sel_hierarchy = 'pima'

#selected countries country_selected can either be 'all' or a list of countries ['Country1','Country']
country_selected = 'all'

#minimum and maximum year to be filtered
min_year = 2000
max_year = 2026

In [0]:
hierarchy_dec = {'Intergovernmental Reform and Coordination':['intergovernmental relations','intergovernmental coordination','intergovernmental reform','local self government'],
             'Functional assignment':['functional assignment','deconcentrated functions','delegated functions','subsidiarity','devolution','devolved functions','expenditure assignment'],
             'Intergovernmental fiscal architecture':['Expenditure assignment','revenue assignment','revenue sharing','equalization','finance commission','grants commission','equitable share','vertical share'],
             'Intergovernmental fiscal transfers':['intergovernmental transfers','fiscal transfers','conditional grant','specific grant','matching grant','performance grant','performance based grant', 'allocation formulae','equalization grant','finance commission','revenue sharing','local government finance','local development grant','earmarked grant'],
             'Locally raised revenue':['own source revenue','business licenses','property tax','licenses and fees','cadaster','local government tax','local revenue raising'],
             'Subnational borrowing':['local government borrowing','local government debt','municipal bonds','city bonds'],
             'Subnational PFM':['subnational PEFA','subnational budgeting'],
             'Local Human Resource Management':['local civil service','local public service'],
             'Institutional Capacity and Performance':['Local Government Performance','District Performance','municipal performance','municipal capacity','local government capacity','local capacity','capacity building grants'],
             'Local Politics':['City council','municipal council','district council','county council','local government council','local council'],
             'Participation and Accountability':['local participation','local accountability','local elections','participatory planning','participatory budgeting'],
             'local infrastructure':['local capital investment planning','local development fund','local development grant','local development plan','local investment fund','urban infrastructure','municipal infrastructure'],'Local Service Delivery':['local government services','local services','results based financing','performance based financing','district health services','district education services','municipal health services','municipal education services','municipal health services']}

hierarchy_pfm = {'PFM Reform and Diagonstic Tools':['Public Expenditure and Financial Accountabilty','PEFA','Country Policy Institutional Assessment','CPIA','PER', 'Public Expenditure Review','PEIR','Public Expenditure and Institutional Review','PFM Reform','PFMR', 'PFM Coordination'],
                 'Transparency, Participation and Accountability':['budget transparency','Open data','public participation','open budget', 'citizens guide to the budget', 'citizens budget', 'participatory budget','BOOST','Buget Portal','Budget Website','open contracting data standards','OCDS','fiscal transparency','participatory budget','participatory based budgeting','pbb','public participation'],
                 'Procurement': ['open contracting data standards','OCDS', 'E-GP','E GP', 'government procurement', 'public procurement', 'e-procurement', 'e procurement', 'procurement reform', 'green procurement'],
                 'Policy-based fiscal strategy and budgeting':['Medium Term Expenditure Framework','MTEF', 'Program-budgeting','PBB','performance budgeting','expenditure plan','Macroeconomic forecasting','Fiscal forecasting','revenue forecasting','revenue projections','Budget formulation','budget documentation','Budget preparation','resource allocation','budget allocation','budget policy','budget plan','Public expenditure','public efficiency','budget cost','gender budgeting','gender budget', 'budget tagging'],
                 'Public Investment and Asset Management':['PIM','Public Investment Appraisal','Asset Management','Infrastructure Governance','Public investment Appraisal','project evaluation','Project Appraisal Criteria','Public Investment Management Assessment','PIMA','Public Investment Plan','PIP','Asset management','asset register'],
                 'Predictability and control in budget execution':['Budget reliability','budget credibility','Budget deviation','Internal audit','Budget execution', 'Integrated Financial Management Information System','IFMIS','cash management','Cash Forecast','Cash Plan','Treasury Single Account','TSA','STA','Single Treasury Account','payment process','electronic funds transfer','EFT'],
                 'Accounting and Reporting':['Budget classification','chart of accounts','budget nomenclature','accounts nomenclature','GFS','COFOG','Financial data integrity','IPSAS','Accrual','accounting standards','Fiduciary assurance','annual financial report','annual financial statement','in-year budget report','budget execution report','bank reconciliation','accounts reconciliation'],
                 'External Scrutiny and Audit':['External audit','parliamentary oversight','legislative scrutiny','supreme audit institution','audit report','cour de comptes','finance commissions','public account commissions','public accuonts committee','parliamentary budget office','court of auditors'],
                 'Financing of Service Delivery' :  ['transfers to sub-national governments','conditional grant','performance grant','equalization grant','allocation formula','fiscal transfer','earmarked grant','specific grant','intergovernmental fiscal','subvention','capitation grant','school grant'],
                 'Climate PFM & PIM' :  ['climate tagging','climate budget tagging','sustainablity reporting','green procurement','green PIM','green public investment management','climate budget','green public procurement','climate tagging','green project appraisal']}

hierarchy_asset_management = {'Key Terms':['asset management', 'infrastructure governance', 'asset register'],
                              'Additional':['asset governance', 'public asset management', 'asset valuation']}

hierarchy_pima = {'Key Terms':['public investment management'],'Additional':['public investment management assessment','climate public investment management']}

hierarchy_drm = {'Key Terms':['domestic revenue mobilization''domestic revenue mobilisation']}

hierarchy_procurement = {'Key Terms':['procurement']}

hierarchy_tagging = {'pfm':hierarchy_pfm,'decentralization':hierarchy_dec,
                     'asset management':hierarchy_asset_management,'pima':hierarchy_pima,
                     'drm':hierarchy_drm,'procurement':hierarchy_procurement}

hierarchy = hierarchy_tagging[sel_hierarchy]
print(hierarchy)

search_term_list = []
for each_group in hierarchy:
  search_term_list.append(hierarchy[each_group])
search_term_list = [item for sublist in search_term_list for item in sublist]

print(search_term_list)

In [0]:
import pandas as pd
from pyspark.sql.functions import countDistinct
import plotly.express as px
import warnings

from pathlib import Path

warnings.simplefilter(action='ignore')

In [0]:
pip install openpyxl

In [0]:
path_prior_actions = 'https://thedocs.worldbank.org/en/doc/ed4dbe34c2cd2555d0d7b28952bddf54-0290032023/original/DPADdatabaseFY22.xlsx'
df_prior_actions = pd.read_excel(path_prior_actions,sheet_name='Prior Actions database')

In [0]:
df_project_master = spark.read.format('csv').option("delimiter","|").option("header","true").load('/Volumes/prd_mega/saiana95/vaiana95/GOAT/PROJECT_MASTER_V3.csv')
df_project_master.createOrReplaceTempView("projMaster")

In [0]:
dbutils.fs.ls('/mnt')

In [0]:
df_project_master_subset = df_project_master.toPandas()

In [0]:
df_project_master_subset.head(2)

In [0]:
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_ID']!='P169212']

df_project_master_subset = df_project_master_subset[df_project_master_subset['LNDNG_INSTR_LONG_NAME'].isin(['Development Policy Lending','Investment Project Financing','Program-for-Results Financing'])]

df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_STAT_NAME'].isin(['Closed','Active'])]

if (country_selected=='all'):
  df_project_master_subset = df_project_master_subset
else:
  df_project_master_subset = df_project_master_subset[df_project_master_subset['CNTRY_SHORT_NAME'].isin(country_selected)] 


df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(str)!='None']
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)>=min_year]
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)<=max_year]

In [0]:
if (sel_projects!='all'):
  df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_ID'].isin(list_projects)]

In [0]:
df_project_master_subset = df_project_master_subset[['PROJ_ID','PROJ_DISPLAY_NAME','PROJ_APPRVL_FY','PROJ_DEV_OBJECTIVE_DESC','RGN_NAME','PROJ_STAT_NAME','CNTRY_SHORT_NAME',
                                                    'PROD_LINE_NAME','LNDNG_INSTR_TYPE_CODE','LNDNG_INSTR_LONG_NAME','CMT_AMT','IBRD_FTF_COMM_AMT',
                                                    'IBRD_FTF_DISB_AMT','PROJECTED_DISB_AMT',
                                                    'PARENT_PROJ_ID','TEAM_LEAD_FULL_NAME','LEAD_GP_NAME','PROJ_SHORT_NAME',
                                                    'PROJ_LGL_NAME','PROD_LINE_TYPE_NAME','DLI_IND','IMPLEMENTING_AGY_NAME']]

In [0]:
df_project_master_subset['PROJ_ID'].nunique()

In [0]:
df_project_master_subset.head(2)

In [0]:
fig1_df = df_project_master_subset.groupby(['PROJ_APPRVL_FY','LNDNG_INSTR_LONG_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'LNDNG_INSTR_LONG_NAME':'Lending Instrument'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Lending Instrument',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.G10)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)


In [0]:
fig1_df = df_project_master_subset.groupby(['PROJ_APPRVL_FY','PROJ_STAT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'PROJ_STAT_NAME':'Project Status'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Project Status',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Set2)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)

In [0]:
fig1_df = df_project_master_subset.groupby(['LNDNG_INSTR_LONG_NAME','PROD_LINE_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'LNDNG_INSTR_LONG_NAME':'Lending Instrument','PROD_LINE_NAME':'Product Line'})
fig1 = px.bar(fig1_df,x='Product Line',y='PROJ_ID',color='Lending Instrument',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.G10)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Product Line Name', tickangle=0, tickfont=dict(size=10))
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=2000,height=650)


In [0]:
df_project_master_subset['PROJ_ID'].nunique()

In [0]:
display(df_project_master_subset.astype(str))

In [0]:
hierarchy

In [0]:
df_pdo_search = df_project_master_subset[['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']]

def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return None
  else:
    return None

for each_group in hierarchy:
  df_pdo_search[each_group] = df_pdo_search['PROJ_DEV_OBJECTIVE_DESC'].apply(check_pfm_categories,category=each_group)

df_pdo_search_selected = df_pdo_search.set_index(['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']).dropna(how='all').reset_index()

display(df_pdo_search_selected.astype(str))

In [0]:
display(pd.merge(df_pdo_search_selected,df_project_master_subset,on=['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']))

In [0]:
sel_projects_pdo = df_project_master_subset[df_project_master_subset['PROJ_ID'].isin(list(df_pdo_search_selected['PROJ_ID'].unique()))]

fig1_df = sel_projects_pdo.groupby(['PROJ_APPRVL_FY','PROJ_STAT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'PROJ_STAT_NAME':'Project Status'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Project Status',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Set2)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)

In [0]:
fig1_df = sel_projects_pdo.groupby(['LEAD_GP_NAME','CNTRY_SHORT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'CNTRY_SHORT_NAME':'Country','LEAD_GP_NAME':'Lead GP'})
fig1 = px.bar(fig1_df,x='Country',y='PROJ_ID',color='Lead GP',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Safe)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year', tickangle=10, tickfont=dict(size=8))
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1500,height=400)

In [0]:
list_projects = df_project_master_subset['PROJ_ID'].unique()

In [0]:
df_prior_actions.head(2)

In [0]:
df_prior_actions[df_prior_actions['Project ID'].isin(list_projects)]['Project ID'].nunique()

In [0]:
display(df_prior_actions[df_prior_actions['Project ID'].isin(list_projects)][['Project ID','TEXT']])

In [0]:
pdo_link1 = '/mnt/dataanalyticslake/WB/OfficialUseOnly/CorporateData/Opeerations/Lending/Results/PROJECT_RESULT_IND_DETAIL.csv'

df_results_indicators = spark.read.format('csv').option("delimiter","|").option("header","true").load(pdo_link1)
df_results_indicators = df_results_indicators.toPandas()

# PDO and Intermediate Results Indicator
df_results_indicators = df_results_indicators[['PROJ_ID','IND_TYPE_NAME','IND_NAME','BASELINE_VAL_TEXT','TGT_VAL_TEXT','UOM_NAME']]
df_results_indicators = df_results_indicators[df_results_indicators['PROJ_ID'].isin(df_project_master_subset['PROJ_ID'].unique())]

In [0]:
%sql
SELECT CNTRY_NAME, COUNT(AGY_USER_KEY) AS user_count FROM qa_data_explorer.dm_ops_gold.procurement_agency_user_v3 WHERE DELTD_IND = 0 GROUP BY CNTRY_NAME ORDER BY user_count DESC LIMIT 1000;
 

In [0]:
df_results_indicators.groupby('IND_TYPE_NAME')['PROJ_ID'].nunique()

In [0]:
display(df_results_indicators.astype(str))

In [0]:
df_results_indicators['PROJ_ID'].nunique()

In [0]:
df_prior_actions_selected = df_prior_actions[df_prior_actions['Project ID'].isin(list_projects)][['Project ID','TEXT']]

def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return None
  else:
    return None

for each_group in hierarchy:
  df_prior_actions_selected[each_group] = df_prior_actions_selected['TEXT'].apply(check_pfm_categories,category=each_group)

df_prior_actions_selected = df_prior_actions_selected.set_index(['Project ID','TEXT']).dropna(how='all').reset_index()

display(df_prior_actions_selected.astype(str))

In [0]:
df_prior_actions_selected['Project ID'].nunique()

In [0]:
display(sel_projects_pa)

In [0]:
sel_projects_pa = df_project_master_subset[df_project_master_subset['PROJ_ID'].isin(list(df_prior_actions_selected['Project ID'].unique()))]

fig1_df = sel_projects_pa.groupby(['PROJ_APPRVL_FY','PROJ_STAT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'PROJ_STAT_NAME':'Project Status'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Project Status',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Set2)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)

In [0]:
fig1_df = sel_projects_pa.groupby(['LEAD_GP_NAME','CNTRY_SHORT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'CNTRY_SHORT_NAME':'Country','LEAD_GP_NAME':'Lead GP'})
fig1 = px.bar(fig1_df,x='Country',y='PROJ_ID',color='Lead GP',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Safe)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year', tickangle=10, tickfont=dict(size=8))
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1500,height=400)

In [0]:
df_prior_actions_selected['Project ID'].nunique()

In [0]:
df_pad_data = spark.sql("select * from VIETNAMML.VMLPADImageBankDocuments")
df_subset = df_pad_data.select('title','fullText','document_type','document_name','lending_instrument','language','project_id','datee')
df_pads_selected = df_subset.toPandas()

df_pads_selected['language'] = df_pads_selected['language'].apply(lambda x: x[0])
df_pads_selected['proj_count'] = df_pads_selected['project_id'].apply(lambda x: len(x))
df_pads_selected = df_pads_selected[df_pads_selected['language']=='English']
df_pads_selected['document_name'] = df_pads_selected['document_name'].apply(lambda x: x[0])
df_pads_selected = df_pads_selected.explode('project_id')

df_pads_selected = df_pads_selected[df_pads_selected['project_id'].isin(df_project_master_subset['PROJ_ID'].unique())]

df_project_master_subset = pd.merge(df_project_master_subset,df_pads_selected,left_on='PROJ_ID',right_on='project_id',how='left')

path_proj_sector = '/mnt/dataanalyticslake/WB/OfficialUseOnly/CorporateData/Operations/CrossCutting/Project/PROJECT_SECTOR.csv'
df_proj_sector = spark.read.format('csv').option("delimiter","|").option("header","true").load(path_proj_sector)
df_proj_sector = df_proj_sector.toPandas()
df_proj_sector = df_proj_sector[df_proj_sector['PROJ_ID'].isin(df_project_master_subset['PROJ_ID'].unique())].groupby('PROJ_ID')['SECT_TEXT'].apply(list).reset_index().dropna()

df_result_set = pd.merge(df_project_master_subset,df_proj_sector,on='PROJ_ID')
df_result_set = df_result_set[df_result_set['fullText'].astype(str)!='nan']

In [0]:
search_term_list

In [0]:
def check_keyword(x,words):
  result_summary = ''
  if isinstance(x, float):
    return 'PAD Not Available'
  else:
    if(len(x)>=24):
      pad_text = x[24]
      pad_text_list = pad_text.split('.')
      for n in range(0,len(pad_text_list)):
        each_sentence = pad_text_list[n]
        res = any(ele in each_sentence for ele in words) 
        if(res==True):
          result_summary = result_summary + pad_text_list[n-1] + '. '
          result_summary = result_summary + each_sentence
      return(str(result_summary).replace(',','/').replace(';',''))
    else:
      return None

df_result_set['search_results_pads'] = df_result_set['fullText'].apply(check_keyword,words=search_term_list)

def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return 'No'
  else:
    return 'No'

for each_group in hierarchy:
  df_result_set[each_group] = df_result_set['search_results_pads'].apply(check_pfm_categories,category=each_group)
  

df_result_set = df_result_set[df_result_set['search_results_pads']!='PAD Not Available']
df_result_set['len_document_type'] = df_result_set['document_type'].apply(lambda x:len(x))
df_result_set['document_type'] = df_result_set['document_type'].apply(lambda x:x[0])

list_cols = []
for each_group in hierarchy:
  list_cols.append(each_group)
for each_val in ['PROJ_ID','document_type']:
  list_cols.append(each_val)
  
df_result_sel = df_result_set[list(list_cols)].set_index(['PROJ_ID','document_type']).replace({'No': None}).dropna(how='all').fillna('No').reset_index().drop_duplicates()

In [0]:
df_result_cleaned = df_result_sel.groupby(['PROJ_ID','document_type']).agg(list)

def clean_data(x):
  if ('Yes' in x):
    return 'Yes'
  else:
    return 'No'
  
for each_col in df_result_cleaned.columns:
  df_result_cleaned[each_col] = df_result_cleaned[each_col].apply(clean_data)
  
df_result_cleaned = df_result_cleaned.reset_index().drop_duplicates()

In [0]:
display(df_result_cleaned.astype(str))#1007

In [0]:
col_names = list(df_results_indicators.columns)

def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return None
  else:
    return None

In [0]:
for each_group in hierarchy:
  df_results_indicators[each_group] = df_results_indicators['IND_NAME'].apply(check_pfm_categories,category=each_group)
  
df_results_indicators_extracted = df_results_indicators.set_index(col_names).dropna(how='all').reset_index()

display(df_results_indicators_extracted.astype(str))
  

In [0]:
l1 = list(df_results_indicators_extracted['PROJ_ID'].unique())
display(sel_projects_pdo[sel_projects_pdo['PROJ_ID'].isin(l1)])